# Moon Rover MPC Testing

In [ ]:
%matplotlib ipympl

import jax

jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_log_compiles", True)

In [ ]:
import functools

import control as ct
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm

from exp_mpc.stewart_min import mp_mpl, mpc_spec, opt, siso, viz

In [ ]:
limits = mpc_spec.MPCLimits()
spec = mpc_spec.MPCSpec()

In [ ]:
# sin wave example

def gen_sin_refs(sin_refs: list[str], amplitude: float=1.0, size=6000):
    rdm_offset = np.random.randint(-1000, 1000, size=6)
    ts = np.arange(size, dtype=float) * spec.dt

    rdm_offset = np.tile(rdm_offset.reshape(1, -1), reps=(size, 1))
    ts = np.tile(ts.reshape(-1, 1), reps=(1, 6))
    ts += rdm_offset
    sin_vals = np.sin(ts) * amplitude

    str_map = dict(zip(["accx", "accy", "accz", "omex", "omey", "omez"], range(6)))
    refs = np.zeros((size, 6))
    refs[:, 2] += mpc_spec.gravity[2]
    for ref in sin_refs:
        idx = str_map[ref]
        refs[:, idx] += sin_vals[:, idx]

    acc_ref = refs[:, :3]
    omega_ref = refs[:, 3:]
    return acc_ref, omega_ref

acc_ref, omega_ref = gen_sin_refs(["accy"], amplitude=1.0)

In [ ]:
begin = 0
# num_steps = 100 * 60 * 3
num_steps = acc_ref.shape[0]
n = 200  # horizon

In [ ]:
# cost setup
# weights = mpc_spec.ExpWeights(  # conservative
#     lin_dyn=jnp.ones(3) * 1e5,
#     omega=jnp.ones(3) * 5e5,
#     control=jnp.ones(6) * 1e-1,
#     alpha_acc=jnp.array([1.0]),
#     alpha_omega=jnp.array([1.0]),
# )
weights = mpc_spec.ExpWeights(  # lander_acc
    lin_dyn=jnp.ones(3) * 1e5,
    omega=jnp.array([1e0, 1e0, 1e2]) * 1e5,
    control=jnp.ones(6) * 1e-3,
    alpha_acc=jnp.array([1.0]),
    alpha_omega=jnp.array([0.0]),
)
# weights = mpc_spec.ExpWeights(  # lander_ome
#     lin_dyn=jnp.ones(3) * 1e5,
#     omega=jnp.array([1e0, 1e0, 1e2]) * 5e5,
#     control=jnp.ones(6) * 1e-1,
#     alpha_acc=jnp.array([0.0]),
#     alpha_omega=jnp.array([0.0]),
# )
limits = mpc_spec.MPCLimits()
spec = mpc_spec.MPCSpec.init_weight_margins(weights, limits, max_iter=2, max_ls=1, use_terminal=True, init_norm=1e-1)
train_step = functools.partial(opt.train_step_with_cost, spec, opt_scheme="jax")

In [ ]:
# run setup
train_state = opt.TrainState.zero_init(spec, acc_ref[0, 2])
train_list = []
times = []
res_list = []

In [ ]:
# precompile
# aref = jnp.tile(acc_ref[0], reps=(n, 1))
# oref = jnp.tile(omega_ref[0], reps=(n, 1))
aref = acc_ref[:n]
oref = omega_ref[:n]
# aref = acc_ref[0]
# oref = omega_ref[0]
print(train_step(train_state, aref, oref)[-1])  # compile + run-time

In [ ]:
# run
for i in tqdm.tqdm(range(num_steps - n)):
    aref = jnp.tile(acc_ref[begin + i], reps=(n, 1))
    oref = jnp.tile(omega_ref[begin + i], reps=(n, 1))
    # aref = acc_ref[begin + i: begin + i + n]
    # oref = omega_ref[begin + i: begin + i + n]
    # aref = acc_ref[begin + i]
    # oref = omega_ref[begin + i]
    train_state, res, t_tot = train_step(train_state, aref, oref)
    train_list.append(train_state)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
# # run zero for a minute
# for i in tqdm.tqdm(range(200 * 60)):
#     aref = jnp.tile(const.moon_gravity, reps=(n, 1))
#     oref = jnp.tile(jnp.zeros(3), reps=(n, 1))
#     train_state, sol, res, t_tot = train_step(aref, oref, train_state)
#     train_list.append(train_state)
#     sol_list.append(sol)
#     res_list.append(res)
#     times.append(t_tot)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
plot_size = len(train_list)
trajectory = train_list[:plot_size]
references = {
    "xyz-acceleration": jnp.array(acc_ref[begin: begin + plot_size]),
    "angular-velocity": jnp.array(omega_ref[begin: begin + plot_size]),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=trajectory, limits=limits, spec=spec, references=references)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=trajectory, limits=limits, spec=spec)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=trajectory, limits=limits, spec=spec)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=trajectory, limits=limits, spec=spec, use_estop=True)

In [ ]:
assert False

## animations

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
mp_mpl.call_mp_animate_trajectory(
    file_name="../data/sms_acc_3d.mp4",
    trajectory=trajectory,
    limits=limits,
    spec=spec,
)